In [ ]:
import pandas as pd
import numpy as np
import os

%load_ext autoreload
%autoreload 2

In [ ]:
path_to_data = "C:/Users/maran/OneDrive/Documents/Git Profile/Data-Projects/dxc_tech/data/data.npy"

state_dict ={
  "AL": "Alabama",
  "AK": "Alaska",
  "AZ": "Arizona",
  "AR": "Arkansas",
  "CA": "California",
  "CO": "Colorado",
  "CT": "Connecticut",
  "DE": "Delaware",
  "FL": "Florida",
  "GA": "Georgia",
  "HI": "Hawaii",
  "ID": "Idaho",
  "IL": "Illinois",
  "IN": "Indiana",
  "IA": "Iowa",
  "KS": "Kansas",
  "KY": "Kentucky",
  "LA": "Louisiana",
  "ME": "Maine",
  "MD": "Maryland",
  "MA": "Massachusetts",
  "MI": "Michigan",
  "MN": "Minnesota",
  "MS": "Mississippi",
  "MO": "Missouri",
  "MT": "Montana",
  "NE": "Nebraska",
  "NV": "Nevada",
  "NH": "New Hampshire",
  "NJ": "New Jersey",
  "NM": "New Mexico",
  "NY": "New York",
  "NC": "North Carolina",
  "ND": "North Dakota",
  "OH": "Ohio",
  "OK": "Oklahoma",
  "OR": "Oregon",
  "PA": "Pennsylvania",
  "RI": "Rhode Island",
  "SC": "South Carolina",
  "SD": "South Dakota",
  "TN": "Tennessee",
  "TX": "Texas",
  "UT": "Utah",
  "VT": "Vermont",
  "VA": "Virginia",
  "WA": "Washington",
  "WV": "West Virginia",
  "WI": "Wisconsin",
  "WY": "Wyoming"
}

csv_dir = 'final_answer'
os.makedirs(csv_dir, exist_ok=True)

In [ ]:
data = np.load(path_to_data, allow_pickle=True)
data.shape

In [ ]:
data[:50]

In [ ]:
df = pd.DataFrame(data, columns=['Raw Name'])
df.head()

In [ ]:
df[['Location','given_abbreviation']] = df['Raw Name'].str.split(',', expand=True)
df['given_abbreviation'] = df['given_abbreviation'].str.upper().str.strip(' ')
df['abbreviation_length'] = df['given_abbreviation'].str.len()
df['location_word_count'] = df['Location'].str.split(' ').str.len()
df['Location'].nunique(), df['given_abbreviation'].nunique()

In [ ]:
df

In [ ]:
state_ab_only_list = list(state_dict.keys())
df['correct_length'] = df['abbreviation_length'] == 2.0
df['has_correct_abbreviation'] = df['given_abbreviation'].isin(state_ab_only_list)
df['valid_state_pair']= df['correct_length'] & df['has_correct_abbreviation']
df.shape

Invalid examples: 'el paraiso, pr', 'popponesset island, na'

In [ ]:
df[df['valid_state_pair'] == False]

In [ ]:
df[(df['correct_length'] == False) & df['has_correct_abbreviation'] == False]

In [ ]:
df[df['valid_state_pair']]

In [ ]:
df['given_abbreviation'].str.contains(r'\d').any()

In [ ]:
total_rows = len(df)
percent_correct_length = df['correct_length'].mean() * 100
percent_has_correct_abbreviation = df['has_correct_abbreviation'].mean() * 100
percent_valid_state_pair = df['valid_state_pair'].mean() * 100

print(f"Percentage with correct length: {percent_correct_length:.2f}%")
print(f"Percentage with correct abbreviation: {percent_has_correct_abbreviation:.2f}%")
print(f"Percentage with valid state pair: {percent_valid_state_pair:.2f}%")

In [ ]:
for_csv = df[df['valid_state_pair']].reset_index(drop=True).loc[:5000, ['Location', 'given_abbreviation']].rename(columns={'given_abbreviation': 'State', 'Location':'City'})
for_csv['City'] = for_csv['City'].str.upper()
if not for_csv['State'].nunique() == 50:
    raise('Error')
for_csv.to_csv(os.path.join(csv_dir, 'city_state_sample.csv'), index=False)